<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/1_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. bad 純粹性


In [ ]:
# --- 在 run_ultimate_rca 循環內 ---

# 1. 計算雙向交叉表
# ct_row: P(Cluster | Category) -> 每一列加起來是 1 (類別內的組成)
ct_row = pd.crosstab(df_proc[col], df_proc[cluster_col], normalize='index')
# ct_col: P(Category | Cluster) -> 每一行加起來是 1 (Bad 資料的去向)
ct_col = pd.crosstab(df_proc[col], df_proc[cluster_col], normalize='columns')

if bad_label in ct_row.columns and bad_label in ct_col.columns:
    # 計算每一類的「雙向純粹度」: P(Bad|Cat) * P(Cat|Bad)
    # 只有當 Bad 同時「集中」且「純淨」時，這個值才會高
    signature_scores = ct_row[bad_label] * ct_col[bad_label]
    max_signature = signature_scores.max()
else:
    max_signature = 0

# 2. 結合原本的相關性與樣本懲罰
# 這樣如果類別數多(n個類別)，max_signature 會自然因為 P(Cat|Bad) 被稀釋而變低
s_purity_ext = max_signature * res['Adj_T'] * c_penalty

**Q : 問題**

那如果說 bad 都很純粹在一個類別裡  他的分數 >  bad都很純粹在兩個類別裡  >> bad很純粹在n個類別裡  上面那個可以做到?  

也想問  bad都很純粹在2-n個類別裡 的分數  比起不純粹在2個類別裡  哪ˇ個分數高?  還是不一定

----

這是一個非常深刻的**機率分佈**與**物理意義**問題。老闆的直覺是對的：**「罪證越集中，兇手越明確」**。

你設計的這個邏輯，在統計學上其實是在衡量「因果關係的**稀疏性 (Sparsity)**」。我設計的這套 `Signature_Purity` 指標完全可以精準達成你的要求，甚至能自動回答你關於「純度 vs 數量」的難題。

---

### 1. 回答你的第一個問題：類別數量（1 vs 2 vs N）

**可以，這套公式會產生自動的「衰減效應」。**

公式為： $P(Bad|Cat) \times P(Cat|Bad)$

也可以看成 precision x recall  = F1-socre

* **Bad 純粹在 1 個類別**：
  * $1.0 \times 1.0 = \mathbf{1.0}$


* **Bad 純粹在 2 個類別**（假設各佔一半）：
  * 類別 A：$1.0 \times 0.5 = 0.5$
  * 類別 B：$1.0 \times 0.5 = 0.5$
  * 最終取 `max`： 0.5


* **Bad 純粹在 N 個類別**：
  * 每個類別：$1.0 \times (1/N) = \mathbf{1/N}$



**結論：** 隨類別數量增加，分數會以  $1/N$ 的速度線性下降。這完美符合你老闆「1個 > 2個 >> N個」的排序邏輯。

---

### 2. 回答你的第二個問題：純粹 vs 數量

**問題：** 「很純粹但在 2 個類別」 vs 「不純粹但在 2 個類別」，誰分數高？

**答案是：前者（很純粹在 2 個類別）分數一定比較高。**

讓我們代入數值來證明：

* **情況 A：很純粹在 2 個類別 (100 壞片在 A, 100 壞片在 B，且 A/B 都沒出過好片)**
  * $P(Bad|A) = 1.0$, $P(A|Bad) = 0.5$
  * 分數：$1.0 \times 0.5 = \mathbf{0.5}$


* **情況 B：不純粹在 2 個類別 (100 壞片在 A，但 A 其實也做了 100 片好片；同樣情形在 B)**
  * $P(Bad|A) = \mathbf{0.5}$, $P(A|Bad) = 0.5$
  * 分數：$0.5 \times 0.5 = \mathbf{0.25}$



**結論：** 因為我們是用「乘法」，所以「純度」只要下降，分數就會呈平方級別地快速衰減。這能確保系統優先抓出那些「只要一出產就是壞片」的故障機台。

---

### 3. 指標視覺化對比 (老闆報告用)

你可以用這張表跟老闆說明你的指標多聰明：

| 壞片分佈狀態 | 分數計算  ($P_{Row} \times P_{Col}$) | 最終得分 | 物理意義 |
| --- | --- | --- | --- |
| **極度鎖定** (1個類別，全壞) | $1.0 \times 1.0$ | **1.0** | **這台機台壞透了，必修。** |
| **分散但危險** (2個類別，全壞) | $1.0 \times 0.5$ | **0.5** | 兩台機台可能共用同一個耗材。 |
| **量大但不純** (1個類別，壞片都在這但好片也很多) | $0.2 \times 1.0$ | **0.2** | 這是製程波動，不是機台完全失效。 |
| **全域污染** (N個類別) | $0.1 \times 0.1$ | **0.01** | **排除機台根因，這是環境或材料問題。** |

